# HotpotQA Baseline

Baseline model using DistilBERT with sentence transformers for retrieval.

In [1]:
%pip install -q -r ../requirements.txt tf-keras


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
import json
import os
import sys
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
from tqdm import tqdm

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.modules['tensorflow'] = None
sys.modules['tf_keras'] = None
sys.modules['keras'] = None

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
    USE_SENTENCE_TRANSFORMERS = True
except ImportError:
    USE_SENTENCE_TRANSFORMERS = False

sys.path.append(str(Path.cwd().parent))

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

Device: mps


## Configuration

In [3]:
CONFIG = {
    'data_dir': Path('../data'),
    'results_dir': Path('../results'),
    'model_name': 'distilbert-base-uncased-distilled-squad',
    'max_length': 512,
    'doc_stride': 128,
    'batch_size': 8,
    'learning_rate': 3e-5,
    'num_epochs': 3,
    'top_k_retrieval': 10,
    'max_context_length': 400,
    'device': device
}

CONFIG['results_dir'].mkdir(exist_ok=True)

## Data Loading

In [4]:
def load_hotpot_data(file_path: Path) -> List[Dict]:
    with open(file_path, 'r') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} examples from {file_path.name}")
    return data

def extract_paragraphs(context: List) -> List[str]:
    paragraphs = []
    for para in context:
        if isinstance(para, list) and len(para) >= 2:
            title = para[0] if para[0] else ''
            sentences = para[1] if isinstance(para[1], list) else []
            text = ' '.join(sentences)
            paragraphs.append(f"{title} {text}")
        elif isinstance(para, dict):
            title = para.get('title', '')
            sentences = para.get('sentences', [])
            text = ' '.join(sentences)
            paragraphs.append(f"{title} {text}")
    return paragraphs

def format_context_for_qa(context: List[Dict]) -> str:
    paragraphs = extract_paragraphs(context)
    return ' '.join(paragraphs)

dev_fullwiki = load_hotpot_data(CONFIG['data_dir'] / 'hotpot_dev_fullwiki_v1.json')
dev_distractor = load_hotpot_data(CONFIG['data_dir'] / 'hotpot_dev_distractor_v1.json')

Loaded 7405 examples from hotpot_dev_fullwiki_v1.json
Loaded 7405 examples from hotpot_dev_distractor_v1.json


## Retrieval

In [5]:
class ImprovedRetriever:
    def __init__(self, top_k: int = 10, use_sentence_transformers: bool = True):
        self.top_k = top_k
        self.paragraphs = []
        self.use_sentence_transformers = use_sentence_transformers and USE_SENTENCE_TRANSFORMERS
        
        if self.use_sentence_transformers:
            self.encoder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
            self.vectors = None
        else:
            self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
            self.vectors = None
        
    def index(self, all_paragraphs: List[str]):
        self.paragraphs = all_paragraphs
        
        if self.use_sentence_transformers:
            batch_size = 32
            self.vectors = []
            for i in tqdm(range(0, len(self.paragraphs), batch_size), desc="Indexing"):
                batch = self.paragraphs[i:i+batch_size]
                batch_vectors = self.encoder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
                self.vectors.append(batch_vectors)
            self.vectors = np.vstack(self.vectors)
        else:
            self.vectors = self.vectorizer.fit_transform(self.paragraphs)
        
    def retrieve(self, query: str, top_k: int = None, exclude_indices: set = None) -> List[Tuple[str, int]]:
        if top_k is None:
            top_k = self.top_k
            
        if self.vectors is None or len(self.paragraphs) == 0:
            return []
        
        if self.use_sentence_transformers:
            query_vec = self.encoder.encode([query], convert_to_numpy=True)
            similarities = cosine_similarity(query_vec, self.vectors)[0]
        else:
            query_vec = self.vectorizer.transform([query])
            similarities = cosine_similarity(query_vec, self.vectors)[0]
        
        top_indices = np.argsort(similarities)[::-1]
        
        if exclude_indices:
            top_indices = [idx for idx in top_indices if idx not in exclude_indices]
        
        top_indices = top_indices[:top_k]
        return [(self.paragraphs[i], i) for i in top_indices]
    
    def retrieve_multi_hop(self, question: str, answer: str = None, num_hops: int = 2, top_k_per_hop: int = 5) -> List[Tuple[str, int]]:
        retrieved = []
        retrieved_indices = set()
        
        query = question
        for hop in range(num_hops):
            hop_results = self.retrieve(query, top_k=top_k_per_hop, exclude_indices=retrieved_indices)
            for para, idx in hop_results:
                if idx not in retrieved_indices:
                    retrieved.append((para, idx))
                    retrieved_indices.add(idx)
            
            if hop < num_hops - 1 and answer:
                query = f"{question} {answer}"
            elif hop < num_hops - 1:
                if retrieved:
                    first_para = retrieved[0][0]
                    query = f"{question} {first_para[:200]}"
        
        return retrieved[:self.top_k]

all_paragraphs_fullwiki = []
paragraph_to_title_fullwiki = {}

for example in dev_fullwiki:
    for para in example['context']:
        if isinstance(para, list) and len(para) >= 2:
            title = para[0] if para[0] else ''
            sentences = para[1] if isinstance(para[1], list) else []
            text = ' '.join(sentences)
            para_text = f"{title} {text}"
        elif isinstance(para, dict):
            title = para.get('title', '')
            sentences = para.get('sentences', [])
            text = ' '.join(sentences)
            para_text = f"{title} {text}"
        else:
            continue
        
        if para_text not in all_paragraphs_fullwiki:
            para_idx = len(all_paragraphs_fullwiki)
            all_paragraphs_fullwiki.append(para_text)
            paragraph_to_title_fullwiki[para_idx] = title

print(f"Building corpus: {len(all_paragraphs_fullwiki)} unique paragraphs")

retriever = ImprovedRetriever(top_k=CONFIG['top_k_retrieval'], use_sentence_transformers=True)
retriever.index(all_paragraphs_fullwiki)

Building corpus: 66573 unique paragraphs


Indexing: 100%|██████████| 2081/2081 [10:43<00:00,  3.24it/s]


## Model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModelForQuestionAnswering.from_pretrained(CONFIG['model_name'])
model.to(CONFIG['device'])

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

## Inference

In [7]:
def is_yes_no_question(question: str) -> bool:
    question_lower = question.lower().strip()
    yes_no_starters = [
        'were', 'was', 'are', 'is', 'did', 'do', 'does', 'have', 'has', 'had',
        'can', 'could', 'will', 'would', 'should', 'may', 'might'
    ]
    return any(question_lower.startswith(starter + ' ') for starter in yes_no_starters)

def extract_sentences_from_paragraph(paragraph: str) -> List[str]:
    import re
    sentences = re.split(r'(?<=[.!?])\s+', paragraph)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

def predict_supporting_facts(question: str, answer: str, retrieved_paragraphs: List[Tuple[str, int]], 
                            paragraph_to_title: Dict[int, str], top_k_sentences: int = 5) -> List[List]:
    if not retrieved_paragraphs or not answer or answer == "noanswer":
        return []
    
    question_words = set(question.lower().split())
    answer_words = set(answer.lower().split())
    
    sentence_scores = []
    
    for para_text, para_idx in retrieved_paragraphs:
        title = paragraph_to_title.get(para_idx, "")
        if not title and para_text:
            parts = para_text.split(' ', 1)
            if len(parts) > 1:
                title = parts[0]
                para_text = parts[1]
        
        sentences = extract_sentences_from_paragraph(para_text)
        
        for sent_idx, sentence in enumerate(sentences):
            sentence_words = set(sentence.lower().split())
            
            question_overlap = len(question_words & sentence_words) / max(len(question_words), 1)
            answer_overlap = len(answer_words & sentence_words) / max(len(answer_words), 1)
            combined_score = question_overlap * 0.4 + answer_overlap * 0.6
            
            if combined_score > 0.1:
                sentence_scores.append((title, sent_idx, combined_score))
    
    sentence_scores.sort(key=lambda x: x[2], reverse=True)
    
    supporting_facts = []
    seen = set()
    for title, sent_idx, score in sentence_scores[:top_k_sentences]:
        key = (title, sent_idx)
        if key not in seen and title:
            supporting_facts.append([title, sent_idx])
            seen.add(key)
    
    return supporting_facts

def predict_answer(question: str, context: str, model, tokenizer, device, max_answer_length: int = 100) -> Tuple[str, float, float]:
    if not context or len(context.strip()) == 0:
        return "noanswer", 0.0, 0.0
    
    is_yes_no = is_yes_no_question(question)
    
    inputs = tokenizer(
        question,
        context,
        truncation=True,
        max_length=CONFIG['max_length'],
        padding='max_length',
        return_tensors='pt'
    ).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits
    
    input_ids = inputs['input_ids'][0]
    sep_token_id = tokenizer.sep_token_id
    question_end = (input_ids == sep_token_id).nonzero(as_tuple=True)[0]
    
    if len(question_end) > 0:
        min_start = question_end[0].item() + 1
    else:
        min_start = 1
    
    valid_start_logits = start_logits[0, min_start:]
    valid_end_logits = end_logits[0, min_start:]
    
    start_idx = torch.argmax(valid_start_logits).item() + min_start
    end_idx = torch.argmax(valid_end_logits).item() + min_start
    
    if end_idx < start_idx:
        end_idx = start_idx
    
    answer_tokens = input_ids[start_idx:end_idx+1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    
    if len(answer) > max_answer_length:
        sentences = answer.split('.')
        if len(sentences) > 0 and len(sentences[0]) <= max_answer_length:
            answer = sentences[0].strip()
        else:
            answer = answer[:max_answer_length].rsplit(' ', 1)[0].strip()
    
    if is_yes_no:
        answer_lower = answer.lower()
        if any(word in answer_lower for word in ['yes', 'true', 'correct', 'same', 'both']):
            answer = "yes"
        elif any(word in answer_lower for word in ['no', 'false', 'incorrect', 'different', 'not']):
            answer = "no"
    
    answer = ' '.join(answer.split()).strip()
    
    if not answer or len(answer) == 0:
        answer = "noanswer"
    
    start_score = torch.softmax(start_logits, dim=1)[0, start_idx].item()
    end_score = torch.softmax(end_logits, dim=1)[0, end_idx].item()
    
    return answer, start_score, end_score

## FullWiki Evaluation

In [8]:
predictions_fullwiki = {
    'answer': {},
    'sp': {}
}

eval_subset = dev_fullwiki[:100]

for example in tqdm(eval_subset, desc="FullWiki evaluation"):
    question = example['question']
    example_id = example['_id']
    
    retrieved_paragraphs_with_idx = retriever.retrieve(question, top_k=5)
    context = ' '.join([para for para, _ in retrieved_paragraphs_with_idx])
    answer, _, _ = predict_answer(question, context, model, tokenizer, CONFIG['device'])
    
    retrieved_paragraphs_multi = retriever.retrieve_multi_hop(question, answer=answer if answer != "noanswer" else None, num_hops=2, top_k_per_hop=5)
    context = ' '.join([para for para, _ in retrieved_paragraphs_multi])
    answer, _, _ = predict_answer(question, context, model, tokenizer, CONFIG['device'])
    
    supporting_facts = predict_supporting_facts(question, answer, retrieved_paragraphs_multi, paragraph_to_title_fullwiki)
    
    predictions_fullwiki['answer'][example_id] = answer
    predictions_fullwiki['sp'][example_id] = supporting_facts

FullWiki evaluation: 100%|██████████| 100/100 [00:18<00:00,  5.52it/s]


## FullWiki Results

In [9]:
pred_file_fullwiki = CONFIG['results_dir'] / 'baseline_fullwiki_predictions.json'
with open(pred_file_fullwiki, 'w') as f:
    json.dump(predictions_fullwiki, f, indent=2)

gold_file_fullwiki = CONFIG['results_dir'] / 'baseline_fullwiki_gold.json'
with open(gold_file_fullwiki, 'w') as f:
    json.dump(eval_subset, f, indent=2)

import subprocess
result = subprocess.run(
    ['python', '../scripts/hotpot_evaluate_v1.py', str(pred_file_fullwiki), str(gold_file_fullwiki)],
    capture_output=True,
    text=True
)
print(result.stdout)

{'em': 0.09, 'f1': 0.192915287327052, 'prec': 0.18886147186147187, 'recall': 0.2253730158730159, 'sp_em': 0.0, 'sp_f1': 0.16953174603174606, 'sp_prec': 0.1288333333333333, 'sp_recall': 0.2616666666666666, 'joint_em': 0.0, 'joint_f1': 0.053527949415071815, 'joint_prec': 0.04067380952380953, 'joint_recall': 0.09930555555555558}



## Distractor Evaluation

In [ ]:
predictions_distractor = {
    'answer': {},
    'sp': {}
}

eval_subset_distractor = dev_distractor[:100]

for example in tqdm(eval_subset_distractor, desc="Distractor evaluation"):
    question = example['question']
    example_id = example['_id']
    
    context = format_context_for_qa(example['context'])
    answer, _, _ = predict_answer(question, context, model, tokenizer, CONFIG['device'])
    
    supporting_facts = []
    if answer and answer != "noanswer":
        question_words = set(question.lower().split())
        answer_words = set(answer.lower().split())
        
        sentence_scores = []
        for para in example['context']:
            if isinstance(para, list) and len(para) >= 2:
                title = para[0] if para[0] else ''
                sentences = para[1] if isinstance(para[1], list) else []
            elif isinstance(para, dict):
                title = para.get('title', '')
                sentences = para.get('sentences', [])
            else:
                continue
            
            for sent_idx, sentence in enumerate(sentences):
                sentence_words = set(sentence.lower().split())
                question_overlap = len(question_words & sentence_words) / max(len(question_words), 1)
                answer_overlap = len(answer_words & sentence_words) / max(len(answer_words), 1)
                combined_score = question_overlap * 0.4 + answer_overlap * 0.6
                
                if combined_score > 0.1:
                    sentence_scores.append((title, sent_idx, combined_score))
        
        sentence_scores.sort(key=lambda x: x[2], reverse=True)
        seen = set()
        for title, sent_idx, score in sentence_scores[:5]:
            key = (title, sent_idx)
            if key not in seen:
                supporting_facts.append([title, sent_idx])
                seen.add(key)
    
    predictions_distractor['answer'][example_id] = answer
    predictions_distractor['sp'][example_id] = supporting_facts

pred_file_distractor = CONFIG['results_dir'] / 'baseline_distractor_predictions.json'
with open(pred_file_distractor, 'w') as f:
    json.dump(predictions_distractor, f, indent=2)

gold_file_distractor = CONFIG['results_dir'] / 'baseline_distractor_gold.json'
with open(gold_file_distractor, 'w') as f:
    json.dump(eval_subset_distractor, f, indent=2)

result = subprocess.run(
    ['python', '../scripts/hotpot_evaluate_v1.py', str(pred_file_distractor), str(gold_file_distractor)],
    capture_output=True,
    text=True
)
print(result.stdout)

Distractor evaluation: 100%|██████████| 100/100 [00:02<00:00, 34.85it/s]

{'em': 0.12, 'f1': 0.1962063492063492, 'prec': 0.19305627705627704, 'recall': 0.22426190476190477, 'sp_em': 0.01, 'sp_f1': 0.26203968253968263, 'sp_prec': 0.1993333333333332, 'sp_recall': 0.4086666666666667, 'joint_em': 0.0, 'joint_f1': 0.0673783171347631, 'joint_prec': 0.050011904761904764, 'joint_recall': 0.13238888888888886}



## 10. Summary and Next Steps

In [11]:
print(f"Results saved:")
print(f"  FullWiki: {pred_file_fullwiki}")
print(f"  Distractor: {pred_file_distractor}")

Results saved:
  FullWiki: ../results/baseline_fullwiki_predictions.json
  Distractor: ../results/baseline_distractor_predictions.json
